# Fashion-MNIST MLP + SVD Fine-tuning（`src` / corrected）

このNotebookは、既存の `03_mlp_svd_finetuning.ipynb` と同じ実験を題材に、**`.py` と `.ipynb` の役割分担**を学ぶための教材です。

- `.py`：別の実験でも繰り返し使う処理を書く場所
- `.ipynb`：今回の実験条件、途中結果、グラフ、考察を書く場所

既存Notebookは参照実装として変更せず、このNotebookでは `nn_compression` パッケージをimportして利用します。

> 実験ロジックは元の03を維持します。元の結果を上書きしないため、出力先は `03_mlp_svd_finetuning_using_src_corrected`。 候補比較では rank ごとに seed を変えない。 原本・using_src は実験履歴として残す。


## 1. import：Notebookから `.py` の処理を呼び出す

`pip install -e .` 済みなので、プロジェクト内の実装を通常のPythonパッケージとしてimportできます。

**元の03との一番大きな違い:**  
元の03では `def set_seed(...)` や `class MNISTMLP(...)` をNotebook内に何度も書いていました。  
この教材版では、同じ処理を `src/nn_compression/` の `.py` に置き、ここから呼び出します。

主な対応は次のとおりです。

| import | 実体のファイル | 元03での扱い |
|---|---|---|
| `MNISTMLP` | `models/mlp.py` | Notebook内で `class MNISTMLP` を定義 |
| `make_two_layer_svd_model`, `retained_energy` | `compression/svd.py` | Notebook内で `def` 定義 |
| `evaluate`, `fit_with_early_stopping` | `training/loops.py`, `training/fit.py` | `train_one_epoch` / `evaluate` / Early Stoppingループをセル内に記述 |
| `count_parameters`, `agreement` など | `metrics/model_comparison.py`, `metrics/mlp_macs.py` | Notebook内で `def` 定義 |
| `get_fashion_mnist_datasets` など | `datasets/fashion_mnist.py` | `datasets.FashionMNIST` / `random_split` / `DataLoader` を直書き |
| `extract_pareto_frontier`, `find_knee_point` など | `selection/pareto.py`, `selection/knee.py` | Notebook内で `def` 定義 |
| `find_project_root`, `get_experiment_dirs`, `set_seed` | `utils/paths.py`, `utils/seed.py` | Notebook内で定義・直書き |

これらは別のNotebookでも使うため、Notebook内では再定義しません。

In [ ]:
# ============================================================
# 【.ipynb側】この実験で使う外部ライブラリ
# ============================================================
import copy
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# 【.py側をimport】元の03ではここに def / class を書いていた
# ============================================================
# SVD圧縮: 元03では SVD / devidetwolayer（現 factorize_linear_layer）/
# make_two_layer_svd_model をセル定義
from nn_compression.compression import (
    make_two_layer_svd_model,  # compression/mlp_svd.py（MLPの層置換）
    retained_energy,           # compression/svd.py（特異値エネルギー）
)

# データ準備: 元03では FashionMNIST / random_split / DataLoader を直書き
from nn_compression.datasets import (
    get_fashion_mnist_datasets,      # datasets/fashion_mnist.py
    make_fashion_mnist_loaders,      # 同上
    split_fashion_mnist_dataset,     # 同上
)

# 評価指標: 元03では各セルで def していた
from nn_compression.metrics import (
    estimate_mlp_macs,       # metrics/mlp_macs.py（784→512→256前提）
    accuracy_drop,           # metrics/model_comparison.py
    agreement,               # metrics/model_comparison.py
    benchmark_inference,
    take_inference_batch,
    count_parameters,        # metrics/model_comparison.py
    logits_rmse,             # metrics/model_comparison.py
    parameters_reduction,    # metrics/model_comparison.py
)

# モデル: 元03では class MNISTMLP(nn.Module): をNotebook内に書いていた
from nn_compression.models import MNISTMLP  # models/mlp.py

# Pareto / knee: 元03では GetPoint などをセル定義していた
# .py化に合わせて、Python標準の snake_case 名へ整理している
from nn_compression.selection import (
    extract_pareto_frontier, # selection/pareto.py
    find_knee_point,         # selection/knee.py
    get_first_point,         # selection/knee.py
    line_equation,           # selection/knee.py
)

# 学習: 元03では train_one_epoch / evaluate / Early Stoppingループを直書き
from nn_compression.training import (
    evaluate,                 # training/loops.py
    fit_with_early_stopping,  # training/fit.py（Early Stopping付き学習）
)

# パス・seed: 元03では find_project_root / set_seed をNotebook内に定義
from nn_compression.utils import (
    find_project_root,     # utils/paths.py
    get_experiment_dirs,   # utils/paths.py
    make_torch_generator,  # utils/seed.py
    set_seed,              # utils/seed.py
)


## 2. 実験条件は `.ipynb` 側に置く

seedを固定する**処理**は再利用できるので `.py` 側ですが、`SEED = 0`という**今回使う値**はNotebook側です。

同様に、learning rate、epoch数、patience、rank候補、実験名は「今回どの条件で試すか」という実験記録なのでNotebookに残します。

### `find_project_root` と `get_experiment_dirs` とは？

どちらも `src/nn_compression/utils/paths.py` にある**パス整理用の小さな関数**です。

1. **`find_project_root(Path.cwd())`**
   - いまいるフォルダから親へさかのぼり、`.git` がある場所を「プロジェクトルート」とみなす
   - 例: Notebookが `notebooks/20_fashion_mnist/mlp/` にあっても、ルートは `D:/dev/nn-compression-svd-dmrg` になる
   - 元の03でも同じ関数をNotebook内に書いていた → 今回は `.py` から呼ぶだけ

2. **`get_experiment_dirs(project_root, dataset_name, experiment_name)`**
   - 実験で使う3つのフォルダをまとめて返す
     - `data_dir` → `プロジェクト/data`
     - `models_dir` → `プロジェクト/models/20_fashion_mnist/実験名`
     - `results_dir` → `プロジェクト/results/20_fashion_mnist/実験名`
   - ついでに `mkdir` もする（なければ作る）
   - 元の03では `models_dir = project_root / "models" / ...` を何行も直書きしていた → それを1関数にまとめた

**Notebook側に残すもの:** `dataset_name` と `experiment_name`（今回の実験の名前）  
**`.py`側に置くもの:** 「ルートを探す」「3フォルダを組み立てて作る」という共通手順

> このNotebookだけ、元03の結果を上書きしないよう  
> `experiment_name = "03_mlp_svd_finetuning_using_src_corrected"` にしています。


In [ ]:
# ============================================================
# 【.ipynb側】今回の実験条件（元の03と同じ値）
# ============================================================
SEED = 0
LEARNING_RATE = 0.001
MAX_EPOCHS = 50
PATIENCE = 5
MIN_DELTA = 1e-4

R1_LIST = [16, 32, 64, 128, 256, 512]  # fc1 の rank 候補
R2_LIST = [16, 32, 64, 128, 256]       # fc2 の rank 候補

DATASET_NAME = "20_fashion_mnist"
# 元03は "03_mlp_svd_finetuning"。結果上書き防止のため別名にする
EXPERIMENT_NAME = "03_mlp_svd_finetuning_using_src_corrected"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 【.py】utils/seed.py の set_seed
# 元03では Notebook内に def set_seed(...) を書いていた
set_seed(SEED)

# 【.py】utils/paths.py
# find_project_root: .git がある場所まで親をたどってプロジェクトルートを返す
# get_experiment_dirs: data / models / results の3パスを作って返す
# 元03では models_dir = project_root / "models" / ... を直書きしていた
project_root = find_project_root(Path.cwd())
data_dir, models_dir, results_dir = get_experiment_dirs(
    project_root,
    DATASET_NAME,
    EXPERIMENT_NAME,
)

print("device:", device)
print("project_root:", project_root)
print("data_dir:", data_dir)
print("models_dir:", models_dir)
print("results_dir:", results_dir)


## 3. Fashion-MNISTの準備

データ取得・分割・DataLoader作成の手順は、同じデータセットを使う実験で繰り返すため `.py` 側です。

- `get_fashion_mnist_datasets`：Fashion-MNISTを取得
- `split_fashion_mnist_dataset`：seed付きで分割
- `make_fashion_mnist_loaders`：DataLoaderを作成

ただし、`[50_000, 5_000, 5_000]`という分割数やbatch sizeは元の03で選んだ**実験条件**なので、引数としてNotebook側に明記します。これにより、処理は再利用しつつ「今回何をしたか」がNotebookだけで確認できます。

In [ ]:
# ============================================================
# 【.py】datasets/fashion_mnist.py を使う
# 元03では FashionMNIST / random_split / DataLoader をこのセルに直書きしていた
# ============================================================
set_seed(SEED)

# 学習用全データ(60k)とテスト(10k)を取得
full_train_dataset, test_dataset = get_fashion_mnist_datasets(
    data_dir,
    download=True,
)

# 【.ipynb側】分割数 (50k / 5k / 5k) は実験条件なので引数で渡す
# 元03と同じ件数・同じ seed
train_dataset, validation_dataset, validation_dataset_rank = (
    split_fashion_mnist_dataset(
        full_train_dataset,
        (50_000, 5_000, 5_000),
        seed=SEED,
    )
)

# 【.ipynb側】batch size も実験条件。shuffle方針は .py 側で固定
train_generator = make_torch_generator(SEED)
loaders = make_fashion_mnist_loaders(
    train_dataset,
    validation_dataset,
    test_dataset,
    train_batch_size=64,
    validation_batch_size=64,
    test_batch_size=1000,
    rank_validation_dataset=validation_dataset_rank,
    train_generator=train_generator,
)

train_loader = loaders["train_loader"]
train_eval_loader = loaders["train_eval_loader"]
validation_loader = loaders["validation_loader"]
validation_loader_rank = loaders["validation_loader_rank"]
test_loader = loaders["test_loader"]

assert (len(train_dataset), len(validation_dataset), len(validation_dataset_rank)) == (
    50_000,
    5_000,
    5_000,
)
print("train size:", len(train_dataset))
print("validation size:", len(validation_dataset))
print("rank-selection validation size:", len(validation_dataset_rank))
print("test size:", len(test_dataset))


## 4. Baselineモデルの学習

`MNISTMLP`の層構成は再利用するため `models/mlp.py` にあります。`fit_with_early_stopping`は、元の03にあったEarly Stoppingループを `training/fit.py` に移した関数です。

一方、Adamを使うこと、learning rate、最大epoch数などは実験者が今回選ぶ条件なのでNotebook側に残します。

`fit_with_early_stopping`は、元の03と同じく各epochで学習・validation評価・train再評価を行い、最良validation lossの重みに戻します。戻り値の履歴をDataFrameにするのは、結果を観察するためのNotebook側の仕事です。

In [ ]:
# ============================================================
# 【.py】MNISTMLP / fit_with_early_stopping を使う
# 元03では:
#   - class MNISTMLP をNotebook内定義
#   - for epoch in range(MAX_EPOCHS): ... Early Stopping を直書き
# ============================================================
set_seed(SEED)

# 【.py】models/mlp.py
model = MNISTMLP().to(device)

# 【.ipynb側】損失・optimizer・学習率は実験条件
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 元03と同じく、モデル初期化後・学習開始直前にも seed を戻す
set_seed(SEED)

# 【.py】training/fit.py
# 中で train_one_epoch / evaluate を呼び、Early Stopping する
# 戻り値: model, best_epoch, best_validation_loss, history など
baseline_fit = fit_with_early_stopping(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    max_epochs=MAX_EPOCHS,   # 【.ipynb】実験条件
    patience=PATIENCE,       # 【.ipynb】実験条件
    min_delta=MIN_DELTA,
    train_eval_loader=train_eval_loader,
)

model = baseline_fit["model"]
baseline_state = copy.deepcopy(model.state_dict())

# 【.ipynb側】履歴を DataFrame / グラフにするのは実験ノートの仕事
df_training_history = pd.DataFrame(baseline_fit["history"])

print("Best epoch:", baseline_fit["best_epoch"])
print(f"Best validation loss: {baseline_fit['best_validation_loss']:.4f}")
display(df_training_history)

fig_learning, axes_learning = plt.subplots(1, 2, figsize=(14, 4))
axes_learning[0].plot(df_training_history["epoch"], df_training_history["train_loss"], label="train")
axes_learning[0].plot(df_training_history["epoch"], df_training_history["validation_loss"], label="validation")
axes_learning[0].set_title("Loss history")
axes_learning[0].set_xlabel("epoch")
axes_learning[0].legend()
axes_learning[0].grid(True, alpha=0.3)

axes_learning[1].plot(df_training_history["epoch"], df_training_history["train_acc"], label="train")
axes_learning[1].plot(df_training_history["epoch"], df_training_history["validation_acc"], label="validation")
axes_learning[1].set_title("Accuracy history")
axes_learning[1].set_xlabel("epoch")
axes_learning[1].legend()
axes_learning[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. SVD rank sweep

SVDでLinear層を置き換える処理は別のrankや別のNotebookでも使うため、`compression/svd.py` の `make_two_layer_svd_model` を呼びます。parameter数、MACs、agreement、logits RMSE、推論時間も `.py` 側の評価関数です。

一方、`R1_LIST`と`R2_LIST`、測定回数、どの指標をDataFrameに保存するかは今回の実験設計です。そのため、rankを総当たりするループと結果レコードの組み立てはNotebook側に置きます。

In [ ]:
# ============================================================
# 【.ipynb側】rank候補の総当たりループと結果表の組み立て
# 【.py側】圧縮・評価・指標計算だけを関数として呼ぶ
# 元03では make_two_layer_svd_model / evaluate / MACs などもセル定義していた
# ============================================================
rank_configs = list(itertools.product(R1_LIST, R2_LIST))  # 【.ipynb】実験条件

# 【.py】metrics/model_comparison.py / metrics/mlp_macs.py
BENCH_WARMUP = 20
BENCH_REPEATS = 2000
inference_batch = take_inference_batch(validation_loader_rank)
baseline_bench = benchmark_inference(
    model,
    device=device,
    warmup=BENCH_WARMUP,
    repeats=BENCH_REPEATS,
    input_batch=inference_batch,
    return_details=True,
)
baseline_time_s = baseline_bench["time_s"]
print(
    "benchmark batch_size",
    baseline_bench["batch_size"],
    "shape",
    baseline_bench["input_shape"],
    "warmup",
    BENCH_WARMUP,
    "repeats",
    BENCH_REPEATS,
)
# 【.py】training/loops.py
baseline_rank_validation_loss, baseline_rank_validation_acc = evaluate(
    model,
    validation_loader_rank,
    criterion,
    device,
)

rank_results = []
for fc1_rank, fc2_rank in rank_configs:
    # 【.py】compression/svd.py（Linearを低ランク2層に置換）
    compressed_model = make_two_layer_svd_model(
        model,
        fc1_rank=fc1_rank,
        fc2_rank=fc2_rank,
    )
    validation_loss, validation_acc = evaluate(
        compressed_model,
        validation_loader_rank,
        criterion,
        device,
    )
    # 【.py】metrics/*
    drop = accuracy_drop(
        baseline_rank_validation_acc,
        validation_acc,
        verbose=False,
    )
    # 【.py】metrics/mlp_macs.py
    _, compressed_macs, compute_reduction = estimate_mlp_macs(
        model,
        fc1_rank,
        fc2_rank,
        verbose=False,
    )
    compressed_time_s = benchmark_inference(
        compressed_model,
        device=device,
        warmup=BENCH_WARMUP,
        repeats=BENCH_REPEATS,
        input_batch=inference_batch,
    )

    # 【.ipynb側】どの列を結果表に入れるかは実験設計
    row = {
        "fc1_rank": fc1_rank,
        "fc2_rank": fc2_rank,
        "parameters": count_parameters(compressed_model),
        "parameters_reduction": parameters_reduction(model, compressed_model),
        "validation_loss": validation_loss,
        "validation_acc": validation_acc,
        "accuracy_drop": drop,
        "compressed_macs": compressed_macs,
        "compute_reduction": compute_reduction,
        "baseline_time_ms": baseline_time_s * 1000,
        "compressed_time_ms": compressed_time_s * 1000,
        "agreement": agreement(model, compressed_model, validation_loader_rank, device),
        "logits_rmse": logits_rmse(model, compressed_model, validation_loader_rank, device),
        "retained_energy_fc1": retained_energy(model.fc1, fc1_rank),
        "retained_energy_fc2": retained_energy(model.fc2, fc2_rank),
    }
    rank_results.append(row)
    print(
        f"r1={fc1_rank:3d}, r2={fc2_rank:3d} | "
        f"Acc={validation_acc:.4f} loss={validation_loss:.4f} | "
        f"params={row['parameters']} MACs={compressed_macs} | "
        f"agreement={row['agreement']:.4f}"
    )

print(f"\n集計完了: {len(rank_results)} 件")


## 6. Pareto frontierとknee候補

「他の点に支配されていない候補を抽出する」という判定ロジックは再利用可能なので、`selection/pareto.py` の `extract_pareto_frontier` を使います。直線から最も遠いknee点の計算も `selection/knee.py` からimportします。

一方、次はこの実験固有なのでNotebook側です。

- baselineよりparameter数が多い候補を除外する条件
- `parameters`と`validation_loss`を選ぶこと
- `MinMaxScaler`でどの列を正規化するか
- グラフの見せ方
- kneeの前後1点をfine-tuning候補にする判断

つまり、`.py`は「計算方法」、`.ipynb`は「今回その方法を何に適用するか」を担当します。

In [ ]:
# ============================================================
# 【.ipynb側】前処理フィルタ・正規化・プロット
# 【.py側】Pareto frontier抽出だけ extract_pareto_frontier に任せる
# 元03では dominated判定の for ループをNotebook内に書いていた
# ============================================================
baseline_params = count_parameters(model)  # 【.py】metrics/model_comparison.py
df_rank_results = pd.DataFrame(rank_results)

# 【.ipynb側】「baselineより大きいparamsを落とす」等は実験判断
df_pareto_input = df_rank_results[
    (df_rank_results["parameters"] <= baseline_params)
    & (df_rank_results["compute_reduction"] > 0)
].copy()

# 【.py】selection/pareto.py
# 両目的とも小さいほど良い場合の非劣後点を返す
df_pareto_view = extract_pareto_frontier(
    df_pareto_input,
    x_column="parameters",
    y_column="validation_loss",
)

# 【.ipynb側】どの列を正規化するかは実験設計
scaler = MinMaxScaler()
df_pareto_view[
    ["parameters_normalize", "validation_loss_normalize"]
] = scaler.fit_transform(
    df_pareto_view[["parameters", "validation_loss"]]
)

# 【.ipynb側】可視化
fig_pareto, axes_pareto = plt.subplots(1, 2, figsize=(16, 4))
axes_pareto[0].scatter(
    df_pareto_view["parameters"],
    df_pareto_view["validation_loss"],
)
axes_pareto[0].set_xlabel("parameters")
axes_pareto[0].set_ylabel("validation_loss")
axes_pareto[0].set_title("Pareto: parameters vs validation_loss")

axes_pareto[1].scatter(
    df_pareto_view["parameters_normalize"],
    df_pareto_view["validation_loss_normalize"],
)
axes_pareto[1].set_xlabel("parameters_normalize")
axes_pareto[1].set_ylabel("validation_loss_normalize")
axes_pareto[1].set_title("Pareto: normalized")
plt.tight_layout()

print("pareto points:", len(df_pareto_view))
display(
    df_pareto_view[
        [
            "fc1_rank",
            "fc2_rank",
            "parameters",
            "validation_loss",
            "parameters_normalize",
            "validation_loss_normalize",
        ]
    ]
)
plt.show()

In [ ]:
# ============================================================
# 【.py】get_first_point / line_equation / find_knee_point
# 元03の GetPoint / GetlinearEquation / GetKneePoint_loop を
# Python標準の snake_case 名へ整理して移植
# 【.ipynb側】knee±1を候補にする、という実験判断はここに残す
# ============================================================

# 【.py】selection/knee.py
# get_first_point: 指定列でソートした先頭点の座標を返す
p0 = get_first_point(
    df_pareto_view,
    "parameters_normalize",
    "parameters_normalize",
    "validation_loss_normalize",
)
p1 = get_first_point(
    df_pareto_view,
    "validation_loss_normalize",
    "parameters_normalize",
    "validation_loss_normalize",
)

# 端点を結ぶ直線から最も遠い点が knee
knee_point, knee_distance = find_knee_point(
    df_pareto_view,
    *line_equation(p0, p1),
    "parameters_normalize",
    "validation_loss_normalize",
)

# 【.ipynb側】kneeの前後1点もFine-tuning候補にする
df_pareto_view = df_pareto_view.sort_values("parameters").reset_index(drop=True)
knee_mask = (
    (df_pareto_view["parameters_normalize"] == knee_point[0])
    & (df_pareto_view["validation_loss_normalize"] == knee_point[1])
)
knee_pos = int(df_pareto_view.index[knee_mask][0])
left = max(knee_pos - 1, 0)
right = min(knee_pos + 1, len(df_pareto_view) - 1)

df_pareto_knee = df_pareto_view.iloc[[knee_pos]].copy()
df_pareto_best = df_pareto_view.iloc[left : right + 1].copy()

print("knee distance:", knee_distance)
print(f"knee_pos={knee_pos}, neighbors iloc[{left}:{right + 1}]")
print("knee:")
display(df_pareto_knee.drop(columns=["model"], errors="ignore"))
print("knee ±1:")
display(df_pareto_best.drop(columns=["model"], errors="ignore"))


## 7. knee ±1候補をFine-tuningする

Fine-tuningでも学習ループ自体は同じなので、再び `training/fit.py` の `fit_with_early_stopping` を使います。これが `.py` を切り出す大きな利点です。BaselineとFine-tuningでEarly Stoppingコードをコピーする必要がありません。

Notebook側には次を残します。

- knee前後のどのモデルを対象にするか
- 候補ごとにoptimizerを新しく作ること
- learning rate
- 候補ごとのseed規則
- 学習結果をどの列で記録するか

候補比較では **同じ SEED** と **同じ DataLoader Generator** に戻す。 rank ごとに seed をずらすと Dropout / shuffle が交絡する。


In [ ]:
# ============================================================
# 【.ipynb側】どの候補をFine-tuningするか、候補ごとの seed / optimizer
# 【.py側】学習ループ本体は fit_with_early_stopping を再利用
# 元03では for re_epoch in range(MAX_EPOCHS): を直書きしていた
# ============================================================
set_seed(SEED)
fine_tuned_records = []

for _, pareto_row in df_pareto_best.iterrows():
    fc1_rank = int(pareto_row["fc1_rank"])
    fc2_rank = int(pareto_row["fc2_rank"])
    # 全 rank の model を表に残さず、必要な候補だけ再構築する
    fine_tuned_model = make_two_layer_svd_model(
        model,
        fc1_rank=fc1_rank,
        fc2_rank=fc2_rank,
    ).to(device)

    candidate_optimizer = optim.Adam(
        fine_tuned_model.parameters(),
        lr=LEARNING_RATE,
    )
    set_seed(SEED)
    train_generator.manual_seed(SEED)
    print(f"\n=== fine-tune r1={fc1_rank}, r2={fc2_rank} ===")

    fit_result = fit_with_early_stopping(
        model=fine_tuned_model,
        train_loader=train_loader,
        val_loader=validation_loader,
        criterion=criterion,
        optimizer=candidate_optimizer,
        device=device,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        train_eval_loader=train_eval_loader,
    )

    fine_tuned_records.append(
        {
            "fc1_rank": fc1_rank,
            "fc2_rank": fc2_rank,
            "best_epoch": fit_result["best_epoch"],
            "best_validation_loss": fit_result["best_validation_loss"],
            "history": fit_result["history"],
            "model": fit_result["model"],
        }
    )

df_fine_tuned_models = pd.DataFrame(fine_tuned_records)
display(df_fine_tuned_models.drop(columns=["history", "model"]))


## 8. Fine-tuning前後の結果表とグラフ

`evaluate`の計算方法は共通なので `.py` 側です。しかし、候補を `Aggressive / Balanced / Conservative` と名付けること、どの列を比較するか、DataFrameやグラフをどう見せるかはこの実験だけの判断です。そのため `.ipynb` 側に置きます。

Fine-tuning後は、Early Stoppingに使った `validation_loader` ではなく、rank選択時と同じ `validation_loader_rank` で再評価します。これも元の03と同じ比較条件です。

In [ ]:
# ============================================================
# 【.ipynb側】候補名付け・比較表・グラフ（実験固有）
# 【.py側】evaluate だけ再利用
# ============================================================
fine_tuning_records = []

for idx, pareto_row in df_pareto_best.iterrows():
    fc1_rank = int(pareto_row["fc1_rank"])
    fc2_rank = int(pareto_row["fc2_rank"])

    # 【.ipynb側】kneeより左/本体/右のラベル（元03と同じ）
    if idx < knee_pos:
        candidate = "Aggressive"
    elif idx == knee_pos:
        candidate = "Balanced"
    else:
        candidate = "Conservative"

    matched = df_fine_tuned_models[
        (df_fine_tuned_models["fc1_rank"] == fc1_rank)
        & (df_fine_tuned_models["fc2_rank"] == fc2_rank)
    ]
    if len(matched) != 1:
        raise ValueError(
            f"Fine-tuning済みモデルが一意に見つかりません: "
            f"r1={fc1_rank}, r2={fc2_rank}, matches={len(matched)}"
        )

    # 【.py】evaluate。比較は Early Stopping用ではなく validation_loader_rank
    loss_after, acc_after = evaluate(
        matched.iloc[0]["model"],
        validation_loader_rank,
        criterion,
        device,
    )
    acc_before = float(pareto_row["validation_acc"])
    loss_before = float(pareto_row["validation_loss"])

    fine_tuning_records.append(
        {
            "candidate": candidate,
            "fc1_rank": fc1_rank,
            "fc2_rank": fc2_rank,
            "acc_before": acc_before,
            "acc_after": acc_after,
            "delta_acc": acc_after - acc_before,
            "loss_before": loss_before,
            "loss_after": loss_after,
            "delta_loss": loss_after - loss_before,
        }
    )

df_fine_tuning_result = pd.DataFrame(fine_tuning_records)
display(df_fine_tuning_result)

# 【.ipynb側】可視化
fig_delta, axes_delta = plt.subplots(1, 2, figsize=(14, 4))
axes_delta[0].plot(df_fine_tuning_result["candidate"], df_fine_tuning_result["delta_acc"], marker="o")
axes_delta[0].set_title("Fine-Tuning delta_acc")
axes_delta[0].set_ylabel("delta_acc")
axes_delta[0].grid(True, alpha=0.3)
axes_delta[1].plot(df_fine_tuning_result["candidate"], df_fine_tuning_result["delta_loss"], marker="o")
axes_delta[1].set_title("Fine-Tuning delta_loss")
axes_delta[1].set_ylabel("delta_loss")
axes_delta[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig_after, axes_after = plt.subplots(1, 2, figsize=(14, 4))
axes_after[0].plot(df_fine_tuning_result["candidate"], df_fine_tuning_result["acc_after"], marker="o")
axes_after[0].set_title("Fine-Tuning acc_after")
axes_after[0].set_ylabel("acc_after")
axes_after[0].grid(True, alpha=0.3)
axes_after[1].plot(df_fine_tuning_result["candidate"], df_fine_tuning_result["loss_after"], marker="o")
axes_after[1].set_title("Fine-Tuning loss_after")
axes_after[1].set_ylabel("loss_after")
axes_after[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 最終モデルの選択とTest評価

Pareto抽出の計算は再び `.py` の `extract_pareto_frontier` を使います。一方、Fine-tuning後の候補から

1. validation lossが小さい
2. 同じならparameter数が少ない
3. さらに同じならvalidation accuracyが高い

という順で最終モデルを選ぶ方針は、今回の実験判断なのでNotebookに明記します。

Testデータはこの最終判断が終わってから一度だけ使います。これはTestへの過学習を避けるための実験設計です。

In [ ]:
# ============================================================
# 【.ipynb側】比較表の組み立て・最終選択ルール
# 【.py側】extract_pareto_frontier / evaluate / count_parameters を再利用
# ============================================================
df_candidate_params = df_pareto_best[
    ["fc1_rank", "fc2_rank", "parameters", "parameters_reduction"]
].copy()
df_after_comparison = df_fine_tuning_result.merge(
    df_candidate_params,
    on=["fc1_rank", "fc2_rank"],
    how="left",
    validate="one_to_one",
)

comparison_records = [
    {
        "model": "Baseline",
        "fc1_rank": "-",
        "fc2_rank": "-",
        "validation_acc": baseline_rank_validation_acc,
        "validation_loss": baseline_rank_validation_loss,
        "parameters": baseline_params,
        "parameters_reduction": 0.0,
    }
]
for _, row in df_after_comparison.iterrows():
    comparison_records.append(
        {
            "model": f"{row['candidate']} after FT",
            "fc1_rank": int(row["fc1_rank"]),
            "fc2_rank": int(row["fc2_rank"]),
            "validation_acc": row["acc_after"],
            "validation_loss": row["loss_after"],
            "parameters": int(row["parameters"]),
            "parameters_reduction": row["parameters_reduction"],
        }
    )

df_model_comparison = pd.DataFrame(comparison_records)
display(df_model_comparison)

# Baselineは圧縮モデルの選択対象から除外
df_final_candidates = df_model_comparison[
    df_model_comparison["model"] != "Baseline"
].copy()

# 【.py】selection/pareto.py を再度利用
df_final_pareto = extract_pareto_frontier(
    df_final_candidates,
    x_column="parameters",
    y_column="validation_loss",
)

# 【.ipynb側】最終1モデルの優先順位（元03と同じ）
df_final_pareto = df_final_pareto.sort_values(
    by=["validation_loss", "parameters", "validation_acc"],
    ascending=[True, True, False],
).reset_index(drop=True)

final_model_result = df_final_pareto.iloc[0]
final_fc1_rank = int(final_model_result["fc1_rank"])
final_fc2_rank = int(final_model_result["fc2_rank"])
final_model = df_fine_tuned_models.loc[
    (df_fine_tuned_models["fc1_rank"] == final_fc1_rank)
    & (df_fine_tuned_models["fc2_rank"] == final_fc2_rank),
    "model",
].iloc[0]

print("Selected final model:")
display(df_final_pareto.iloc[[0]])
print(f"Final model: r1={final_fc1_rank}, r2={final_fc2_rank}")

# 最終選択後にTestを評価
final_test_loss, final_test_acc = evaluate(
    final_model,
    test_loader,
    criterion,
    device,
)
baseline_model_for_test = MNISTMLP().to(device)
baseline_model_for_test.load_state_dict(baseline_state)
baseline_test_loss, baseline_test_acc = evaluate(
    baseline_model_for_test,
    test_loader,
    criterion,
    device,
)

df_test_comparison = pd.DataFrame(
    [
        {
            "model": "Baseline",
            "fc1_rank": "-",
            "fc2_rank": "-",
            "parameters": baseline_params,
            "test_loss": baseline_test_loss,
            "test_acc": baseline_test_acc,
            "delta_test_acc": 0.0,
            "delta_test_loss": 0.0,
        },
        {
            "model": "Final (compressed + FT)",
            "fc1_rank": final_fc1_rank,
            "fc2_rank": final_fc2_rank,
            "parameters": count_parameters(final_model),
            "test_loss": final_test_loss,
            "test_acc": final_test_acc,
            "delta_test_acc": final_test_acc - baseline_test_acc,
            "delta_test_loss": final_test_loss - baseline_test_loss,
        },
    ]
)
display(df_test_comparison)

## 10. 結果の保存

CSV名、PNG名、保存する列、モデル名はこの実験に固有なのでNotebook側です。`.py`側は「どう計算するか」を提供しますが、「今回の結果をどんな名前で残すか」は決めません。

このNotebookでは元の03を上書きしないよう、`results/20_fashion_mnist/03_mlp_svd_finetuning_using_src/`と対応するmodelsディレクトリへ保存します。

In [ ]:
# ============================================================
# 【.ipynb側】何をどのファイル名で残すか（実験固有）
# 【.py側】results_dir / models_dir は get_experiment_dirs が用意済み
# ============================================================
# 元の03と同じFine-tuning関連成果物を保存
fine_tuning_csv = results_dir / "fine_tuning_result.csv"
model_comparison_csv = results_dir / "model_comparison.csv"
selected_final_csv = results_dir / "selected_final_model.csv"

df_fine_tuning_result.to_csv(fine_tuning_csv, index=False)
df_model_comparison.to_csv(model_comparison_csv, index=False)
pd.DataFrame([final_model_result]).to_csv(selected_final_csv, index=False)

pareto_figure_path = results_dir / "parameters_vs_validation_loss_normalize.png"
delta_figure_path = results_dir / "fine_tuning_delta.png"
after_figure_path = results_dir / "fine_tuning_after.png"
fig_pareto.savefig(pareto_figure_path, dpi=150, bbox_inches="tight")
fig_delta.savefig(delta_figure_path, dpi=150, bbox_inches="tight")
fig_after.savefig(after_figure_path, dpi=150, bbox_inches="tight")

saved_model_paths = []
for _, row in df_fine_tuned_models.iterrows():
    r1 = int(row["fc1_rank"])
    r2 = int(row["fc2_rank"])
    matched = df_fine_tuning_result[
        (df_fine_tuning_result["fc1_rank"] == r1)
        & (df_fine_tuning_result["fc2_rank"] == r2)
    ]
    tag = str(matched.iloc[0]["candidate"]).lower()
    model_path = models_dir / f"r1-{r1}_r2-{r2}_{tag}_finetuned.pt"
    torch.save(row["model"].state_dict(), model_path)
    saved_model_paths.append(model_path)

final_model_path = (
    models_dir / f"r1-{final_fc1_rank}_r2-{final_fc2_rank}_final.pt"
)
torch.save(final_model.state_dict(), final_model_path)

for path in [
    fine_tuning_csv,
    model_comparison_csv,
    selected_final_csv,
    pareto_figure_path,
    delta_figure_path,
    after_figure_path,
    *saved_model_paths,
    final_model_path,
]:
    print("saved:", path)


## このNotebookでの役割分担

| `.py` に置くもの | `.ipynb` に置くもの |
|---|---|
| 再利用する処理 | 今回だけの実験条件 |
| `MNISTMLP`などのモデル定義 | rank候補 `R1_LIST`, `R2_LIST` |
| SVD分解・Linear層の置換 | learning rate、epoch数、patience |
| 1 epoch学習、評価、Early Stopping | optimizerの選択と作成 |
| seed固定、DataLoader作成 | split数、batch size |
| parameter数、agreement、RMSE、推論時間 | rank sweepの結果レコード・DataFrame |
| Pareto判定、knee距離計算 | 正規化する列、グラフ、CSV名 |
| 同じ入力に同じ計算を行う関数 | 最終rankの採用判断と実験考察 |

### 判断の目安

- 別のNotebookへコピーしたくなった処理は、`.py`に置く候補です。
- 値を変えて比較したい条件や、途中経過を見たい処理は、`.ipynb`に置く候補です。
- `.py`は「道具箱」、`.ipynb`は「その道具を使った実験ノート」と考えると整理しやすくなります。

### 元の03との実験ロジック確認

維持している条件：

- seedは0、決定論設定も同じ
- splitは50,000 / 5,000 / 5,000
- batch sizeはtrain/validationが64、testが1,000
- MLP構造、Adam、learning rate 0.001
- max epochs 50、patience 5、min delta `1e-4`
- rank候補、rank sweep指標、Pareto条件、正規化、knee ±1
- 候補別seed、Fine-tuning、最終選択順、最後のTest評価

意図的な差は次の2点だけです。

1. 重複関数をNotebook内で定義せず、`nn_compression`からimportする
2. 元の実験結果を上書きしないよう、出力先のexperiment名に `_using_src` を付ける

セル内の表示方法や変数名は教材として読みやすく整理していますが、学習・圧縮・選択の数値ロジックは変更していません。